# Comparação de configurações do pipeline

Cada configuração foi rodada sobre as 20 perguntas de `data/eval/questions.jsonl`.
As respostas ficam em cache em `data/eval/runs/`. Este notebook recalcula as
métricas determinísticas a partir do cache (não chama o LLM) e compara.

Análise completa em [`docs/avaliacao.md`](../docs/avaliacao.md).

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.evaluate import load_questions, score

RUNS = Path.cwd().parent / "data" / "eval" / "runs"
items = load_questions()

# arquivo de cache -> rótulo legível
CONFIGS = {
    "d52c779075.jsonl":    "estrutural · hybrid · sem filtro",
    "d52c779075_yr.jsonl": "estrutural · hybrid · ano oráculo",
    "f7d1915fe7_yr.jsonl": "estrutural · bm25 · ano oráculo",
    "f7d1915fe7.jsonl":    "estrutural · bm25 · ano inferido",
    "d550831e81.jsonl":    "fixo · bm25 · ano inferido",
}

In [ ]:
def metrics(fname):
    res = [json.loads(l) for l in (RUNS / fname).read_text(encoding="utf-8").splitlines()]
    rows = score(items, res)
    ans = [r for r in rows if r["answerable"]]
    noans = [r for r in rows if not r["answerable"]]
    return {
        "retrieval hit@k": sum(r["retrieval_hit"] for r in ans) / len(ans),
        "MRR": sum(r["mrr"] for r in ans) / len(ans),
        "resposta correta": sum(r["answer_ok"] for r in ans) / len(ans),
        "abstenção correta": sum(r["answer_ok"] for r in noans) / len(noans),
        "tempo médio (s)": sum(r["seconds"] for r in rows) / len(rows),
    }

df = pd.DataFrame({label: metrics(f) for f, label in CONFIGS.items()}).T
df.round(2)

In [ ]:
ax = df[["retrieval hit@k", "resposta correta"]].plot.barh(
    figsize=(8, 4), xlim=(0, 1))
ax.set_title("Retrieval vs. acurácia da resposta por configuração")
ax.invert_yaxis()

In [ ]:
# acurácia por categoria, na config final
res = [json.loads(l) for l in (RUNS / "d550831e81.jsonl").read_text(encoding="utf-8").splitlines()]
rows = score(items, res)
cat = pd.DataFrame(rows).groupby("category").agg(
    n=("id", "count"),
    retrieval=("retrieval_hit", lambda s: s.dropna().mean()),
    resposta_ok=("answer_ok", "mean"),
)
cat.round(2)

## Conclusões

1. **BM25 > híbrido** para esta base — o embedding multilíngue pequeno é
   grosseiro demais para termos contábeis; a fusão dilui o bom ranking lexical.
2. **Inferir o ano da pergunta** recupera tanto quanto o filtro manual.
3. **Chunking fixo > estrutural** — blocos uniformes ajudam o modelo de 3B;
   a hipótese inicial ("estrutural mantém a tabela inteira") não se confirmou.
4. **Abstenção 100%** em todas as configs — o sistema nunca inventa.
5. O gargalo restante é o LLM lendo tabela (categoria `balanco` é a pior).